In [2]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [3]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [4]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               'Temperature', 'DewPoint', 'v10n', 
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

#y_test

((2732, 19), (2732,), (171, 19), (171,))

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(GradientBoostingRegressor.__module__)

sklearn.ensemble._gb


In [5]:
gbr_base_WM = GradientBoostingRegressor(
    n_estimators=1800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

gbr_base_WM.fit(X_train_scaled, y_train)
y_pred = gbr_base_WM.predict(X_test_scaled)
print("R² for DOC for log scaled | Baseline GB Regressor | WM:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | Baseline GB Regressor | WM:", r2_score(y_test_real, y_pred_orig))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC for log scaled | Baseline GB Regressor | WM: 0.6834426853561297
R² for DOC on normal scaled | Baseline GB Regressor | WM: 0.5742313214591405
R² for DOC on normal: 0.5742313214591405
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5742
MSE  : 1.3277
RMSE : 1.1523
MAE  : 0.6867


In [7]:
#HyperParam Tuning
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, make_scorer
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np
import optuna

#kf = KFold(n_splits=3, shuffle=True, random_state=42)

In [8]:

def objective_GBRcv_WM(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 600, 2100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_state": 42
    }

    model = GradientBoostingRegressor(**params)

    # Repeated 5-fold CV, 3 repeats to reduce variance
    #rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
    r2_scorer = make_scorer(r2_score)

    scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring=r2_scorer)
    mean_r2 = np.mean(scores)
    return mean_r2

In [9]:
study_gbr_WM = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="GBR_withCV_WM"
)

study_gbr_WM.optimize(objective_GBRcv_WM, n_trials=50, show_progress_bar=True, n_jobs=-1)

[I 2026-05-31 17:59:10,219] A new study created in memory with name: GBR_withCV_WM
Best trial: 10. Best value: 0.955027:   2%|▏         | 1/50 [00:24<20:16, 24.82s/it]

[I 2026-05-31 17:59:35,035] Trial 10 finished with value: 0.9550274950343193 and parameters: {'n_estimators': 672, 'learning_rate': 0.04465283583429327, 'max_depth': 4, 'min_samples_split': 3, 'min_samples_leaf': 7, 'subsample': 0.6574258305045101}. Best is trial 10 with value: 0.9550274950343193.
[I 2026-05-31 17:59:35,085] Trial 7 finished with value: 0.759764699850364 and parameters: {'n_estimators': 658, 'learning_rate': 0.013234873176061283, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 2, 'subsample': 0.9436878951009382}. Best is trial 10 with value: 0.9550274950343193.


Best trial: 10. Best value: 0.955027:   6%|▌         | 3/50 [00:42<10:11, 13.02s/it]

[I 2026-05-31 17:59:52,808] Trial 15 finished with value: 0.9400219800854899 and parameters: {'n_estimators': 1228, 'learning_rate': 0.040396616861105336, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 6, 'subsample': 0.6818560585087862}. Best is trial 10 with value: 0.9550274950343193.


Best trial: 10. Best value: 0.955027:   8%|▊         | 4/50 [00:43<06:41,  8.73s/it]

...

[I 2026-05-31 18:04:06,015] Trial 46 finished with value: 0.9802239200982331 and parameters: {'n_estimators': 1834, 'learning_rate': 0.025589470924868647, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 3, 'subsample': 0.7540852140093028}. Best is trial 35 with value: 0.981530736100073.


Best trial: 35. Best value: 0.981531:  98%|█████████▊| 49/50 [04:56<00:02,  2.58s/it]

[I 2026-05-31 18:04:07,044] Trial 47 finished with value: 0.9804885842354993 and parameters: {'n_estimators': 1857, 'learning_rate': 0.026224769048508678, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 3, 'subsample': 0.7579969906259203}. Best is trial 35 with value: 0.981530736100073.


Best trial: 35. Best value: 0.981531: 100%|██████████| 50/50 [04:57<00:00,  5.96s/it]

[I 2026-05-31 18:04:08,108] Trial 49 finished with value: 0.9654030530402499 and parameters: {'n_estimators': 1744, 'learning_rate': 0.010228463559691519, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 3, 'subsample': 0.7701042278024717}. Best is trial 35 with value: 0.981530736100073.


In [10]:
print(study_gbr_WM.best_params)
print(study_gbr_WM.best_value)

{'n_estimators': 1866, 'learning_rate': 0.03338406968671982, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 3, 'subsample': 0.7454280398307638}
0.981530736100073


In [11]:
best_params_WM = study_gbr_WM.best_params
print(best_params_WM)

{'n_estimators': 1866, 'learning_rate': 0.03338406968671982, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 3, 'subsample': 0.7454280398307638}


In [12]:
best_params = best_params_WM

best_gbr_WM = GradientBoostingRegressor(
    **best_params,
    random_state=42
)

best_gbr_WM.fit(X_train_scaled, y_train)
y_pred = best_gbr_WM.predict(X_test_scaled)

print("R² for DOC for log scaled | GBR With Meteo:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | GBR With meteo:", r2_score(y_test_real, y_pred_orig))



# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))


# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R² for DOC for log scaled | GBR With Meteo: 0.6706543028759859
R² for DOC on normal scaled | GBR With meteo: 0.5385103841547108
R² for DOC on normal: 0.5385103841547108
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5385
MSE  : 1.4391
RMSE : 1.1996
MAE  : 0.7045


In [ ]:
#NoMeteo

In [6]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [7]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [8]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features_noMeteo = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               
               #'Temperature', 'DewPoint', 'v10n', #'Precipitation(mm)',
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features_noMeteo]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features_noMeteo]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2732, 16), (2732,), (171, 16), (171,))

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(GradientBoostingRegressor.__module__)

sklearn.ensemble._gb


In [20]:
gbr_base_NM = GradientBoostingRegressor(
     n_estimators=1800,
    learning_rate=0.03,
    max_depth=4,
    subsample=1.0,
    random_state=42,
)

gbr_base_NM.fit(X_train, y_train)
y_pred = gbr_base_NM.predict(X_test)
print("R² for DOC for log scaled | Baseline GB Regressor | NM:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | Baseline GB Regressor | NM:", r2_score(y_test_real, y_pred_orig))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC for log scaled | Baseline GB Regressor | NM: 0.6423082688185041
R² for DOC on normal scaled | Baseline GB Regressor | NM: 0.5200194422764177
R² for DOC on normal: 0.5200194422764177
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5200
MSE  : 1.4968
RMSE : 1.2234
MAE  : 0.6958


In [11]:
#HyperParam Tuning
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, make_scorer
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np
import optuna

#kf = KFold(n_splits=3, shuffle=True, random_state=42)

In [12]:
def objective_GBRcv_NM(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 600, 2100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_state": 42
    }

    model = GradientBoostingRegressor(**params)
    r2_scorer = make_scorer(r2_score)

    scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring=r2_scorer)
    mean_r2 = np.mean(scores)
    return mean_r2

In [13]:
study_gbr_NM = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="GBR_withCV_NM"
)

study_gbr_NM.optimize(objective_GBRcv_NM, n_trials=50, show_progress_bar=True, n_jobs=-1)

[I 2026-05-31 18:36:17,805] A new study created in memory with name: GBR_withCV_NM
Best trial: 11. Best value: 0.797734:   2%|▏         | 1/50 [00:24<19:52, 24.33s/it]

[I 2026-05-31 18:36:42,104] Trial 11 finished with value: 0.7977342465952196 and parameters: {'n_estimators': 791, 'learning_rate': 0.04771608710648337, 'max_depth': 2, 'min_samples_split': 3, 'min_samples_leaf': 6, 'subsample': 0.678229146142738}. Best is trial 11 with value: 0.7977342465952196.


Best trial: 11. Best value: 0.797734:   4%|▍         | 2/50 [00:24<08:11, 10.24s/it]

[I 2026-05-31 18:36:42,499] Trial 4 finished with value: 0.7526276866695691 and parameters: {'n_estimators': 825, 'learning_rate': 0.031488811537033784, 'max_depth': 2, 'min_samples_split': 3, 'min_samples_leaf': 7, 'subsample': 0.6626847498778725}. Best is trial 11 with value: 0.7977342465952196.


Best trial: 11. Best value: 0.797734:   6%|▌         | 3/50 [00:25<04:38,  5.93s/it]

...

[I 2026-05-31 18:39:28,797] Trial 45 finished with value: 0.9711124631542657 and parameters: {'n_estimators': 1284, 'learning_rate': 0.04964999195191972, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 9, 'subsample': 0.6520213630557925}. Best is trial 27 with value: 0.9752760211411627.


Best trial: 27. Best value: 0.975276:  96%|█████████▌| 48/50 [03:11<00:02,  1.25s/it]

[I 2026-05-31 18:39:29,188] Trial 49 finished with value: 0.9745128869386618 and parameters: {'n_estimators': 1233, 'learning_rate': 0.03763150657251105, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.643028855699224}. Best is trial 27 with value: 0.9752760211411627.


Best trial: 27. Best value: 0.975276: 100%|██████████| 50/50 [03:11<00:00,  3.84s/it]

[I 2026-05-31 18:39:29,634] Trial 48 finished with value: 0.974320173423724 and parameters: {'n_estimators': 1265, 'learning_rate': 0.049057264003382554, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.6412019033451641}. Best is trial 27 with value: 0.9752760211411627.
[I 2026-05-31 18:39:29,694] Trial 47 finished with value: 0.9735370307791021 and parameters: {'n_estimators': 1260, 'learning_rate': 0.049751016925567264, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.6472178528422614}. Best is trial 27 with value: 0.9752760211411627.


In [14]:
print(study_gbr_NM.best_params)
print(study_gbr_NM.best_value)

{'n_estimators': 1448, 'learning_rate': 0.04953110623778672, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 1, 'subsample': 0.6216638078288265}
0.9752760211411627


In [15]:
best_params_NM = study_gbr_NM.best_params
print(best_params_NM)

{'n_estimators': 1448, 'learning_rate': 0.04953110623778672, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 1, 'subsample': 0.6216638078288265}


In [16]:
best_params = best_params_NM

best_gbr_NM = GradientBoostingRegressor(
    **best_params,
    random_state=42
)

best_gbr_NM.fit(X_train_scaled, y_train)
y_pred = best_gbr_NM.predict(X_test_scaled)

print("R² for DOC for log scaled | GBR With no Meteo:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | GBR With no meteo:", r2_score(y_test_real, y_pred_orig))



# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))


# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R² for DOC for log scaled | GBR With no Meteo: 0.6488844034170084
R² for DOC on normal scaled | GBR With no meteo: 0.5182612472419021
R² for DOC on normal: 0.5182612472419021
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5183
MSE  : 1.5023
RMSE : 1.2257
MAE  : 0.7076


In [ ]:
#Completed